# Experiment 14: Model-Wide Inactive Weights DBSCAN 3D Tensor Compression

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Core Architecture & Innovations:
1. **Model-Wide Inactive Subspace Profiling**:
   - While Experiments 11–13 compressed the active subspace ($2,400$ coords) and left the remaining ~$4,512$ coords dense, this benchmark compresses the **inactive subspace** across all 26 layers.
   - Partitions each of the 78 submodules into Active ($2,400$) and Inactive ($4,512$) in FP32 precision.
2. **Fine-Scale Inactive DBSCAN on CPU**:
   - Each submodule runs fine-precision DBSCAN ($\\epsilon_{\\text{inact}} = \\max(10^{-4}, 0.18 \\times \\operatorname{std}(v_{\\text{inact}}))$) to group low-activation coordinates into dense semantic clusters.
   - Inactive superweights and density noise (`-1`) are quarantined in FP32.
3. **Inactive 3D Tensors**:
   - Assembles 10 uniform chunks of size 400 into 3D tensors: $\\mathcal{T}_{\\text{inact}} \\in \\mathbb{R}^{10 \\times 400 \\times 1152}$ ($4,000$ coordinates factorized per matrix).
4. **Multi-Tier Inactive Compression Sweeps**:
   - **Tier 1: Inactive Moderate** (`[6, 120, 350]`): Eliminates **~304 Million parameters** across the model.
   - **Tier 2: Inactive Aggressive** (`[4, 80, 200]`): Eliminates **~334 Million parameters** across the model.
   - **Tier 3: Inactive Ultra-Aggressive** (`[3, 40, 100]`): Eliminates **~350 Million parameters** across the model.
5. **Zero-VRAM Safety**:
   - All SVD and Adam GD operations execute on CPU (`device="cpu"`). Evaluation runs with `logits_to_keep=1`.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Ensure fast offline loading from local cache
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    print("Successfully imported ModelManagementInterface from neural_decomp")
except ImportError:
    repo_root = Path.cwd().resolve()
    while repo_root.parent != repo_root:
        if (repo_root / "neural_decomp").is_dir():
            sys.path.insert(0, str(repo_root))
            break
        repo_root = repo_root.parent
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    print("Imported ModelManagementInterface after updating sys.path")

print("TensorLy Backend:", tl.get_backend())
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully imported ModelManagementInterface from neural_decomp
TensorLy Backend: pytorch
PyTorch Version: 2.14.0+cu130
CUDA Available: True
Device Name: NVIDIA GeForce RTX 3070 Ti


In [2]:
# =====================================================================
# STEP 2: Model & Dataset Loading
# =====================================================================
MODEL_ID = "google/gemma-3-1b-it"
NUM_LAYERS = 26
NUM_EVAL_SAMPLES = 1000

print(f"Loading model: {MODEL_ID} on device 0...")
mmi = ModelManagementInterface(
    model_id=MODEL_ID,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

# Cache all 26 layers' pristine weights on CPU
W_orig_all = {}
for l in range(NUM_LAYERS):
    layer_mlp = model.model.layers[l].mlp
    W_orig_all[l] = {
        "gate_proj": layer_mlp.gate_proj.weight.data.clone().cpu(),
        "up_proj":   layer_mlp.up_proj.weight.data.clone().cpu(),
        "down_proj": layer_mlp.down_proj.weight.data.clone().cpu(),
    }

print(f"Cached all {NUM_LAYERS} layers pristine weights on CPU (78 projection matrices).")

# Load GLUE MNLI dataset
print(f"\nLoading GLUE MNLI dataset (150 evaluation samples)...")
dataset = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
eval_data = dataset.select(range(NUM_EVAL_SAMPLES))

label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
label_tokens = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + tok, add_special_tokens=False)[0] for tok in label_tokens]
print(f"Candidate label token IDs: {list(zip(label_tokens, label_token_ids))}")


Loading model: google/gemma-3-1b-it on device 0...


Loading weights: 100%|██████████| 340/340 [00:01<00:00, 190.58it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


NotImplementedError: Cannot copy out of meta tensor; no data!

time: 5.59s
cummulative_time: 8.54s


In [ ]:
# =====================================================================
# STEP 3: Model-Wide Tri-Hook Profiling & Pristine Baseline Accuracy
# =====================================================================
layer_acts = {l: {"gate_proj": [], "up_proj": [], "down_proj": []} for l in range(NUM_LAYERS)}
hooks = []

for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp

    def make_gate_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["gate_proj"].append(
            out.detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    def make_up_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["up_proj"].append(
            out.detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    def make_down_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["down_proj"].append(
            inp[0].detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    hooks.append(lmod.act_fn.register_forward_hook(make_gate_hook(l)))
    hooks.append(lmod.up_proj.register_forward_hook(make_up_hook(l)))
    hooks.append(lmod.down_proj.register_forward_hook(make_down_hook(l)))

baseline_preds, ground_truths = [], []
model.eval()
print(f"Running baseline profiling pass across all {NUM_LAYERS} layers (150 samples)...")
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        baseline_preds.append(pred_label)
        ground_truths.append(sample["label"])

for h in hooks:
    h.remove()

baseline_accuracy = accuracy_score(ground_truths, baseline_preds)
print(f"\nPristine Baseline Accuracy across all 26 layers: {baseline_accuracy * 100:.2f}%")

# Aggregate activation matrices per layer & submodule (Shape: [150, 6912])
acts_matrix_all = {l: {} for l in range(NUM_LAYERS)}
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        acts_matrix_all[l][sub_name] = torch.stack(layer_acts[l][sub_name], dim=0).numpy()

print(f"Captured activation statistics for all 78 submodules ({acts_matrix_all[0]['gate_proj'].shape}).")


In [ ]:
# =====================================================================
# STEP 4: Independent Inactive DBSCAN Pre-Clustering (All 78 Submodules)
# =====================================================================
NUM_ACTIVE = 2400
INACT_CHUNK_SIZE = 400
INACT_NUM_CHUNKS = 10  # 4,000 inactive coords factorized

layer_inactive_data = {}

def cluster_inactive_submodule(acts_matrix, weight_tensor, is_col=False):
    v_all = np.mean(np.abs(acts_matrix), axis=0)
    var_all = np.var(acts_matrix, axis=0)
    
    sorted_all = np.argsort(v_all)
    inactive_pool = sorted_all[:-NUM_ACTIVE]
    active_pool = sorted_all[-NUM_ACTIVE:]
    
    # Notebook 02 Inactive Preprocessing: SVD 95% Denoising + 60% Sparsification
    W_sub_inact = weight_tensor[:, inactive_pool].T if is_col else weight_tensor[inactive_pool, :]
    U, S, Vh = torch.linalg.svd(W_sub_inact, full_matrices=False)
    cum_e = torch.cumsum(S**2, dim=0) / torch.sum(S**2)
    r95 = (cum_e >= 0.95).nonzero()[0].item() + 1
    W_denoised = U[:, :r95] @ torch.diag(S[:r95]) @ Vh[:r95, :]
    eps_val = torch.quantile(torch.abs(W_denoised), 0.60)
    W_sparse = W_denoised.clone()
    W_sparse[torch.abs(W_sparse) < eps_val] = 0.0
    
    W_clean = weight_tensor.clone()
    if is_col:
        W_clean[:, inactive_pool] = W_sparse.T
    else:
        W_clean[inactive_pool, :] = W_sparse

    v_inact = v_all[inactive_pool]
    eps_inact = max(1e-4, float(np.std(v_inact) * 0.18))
    
    db = DBSCAN(eps=eps_inact, min_samples=30, metric="euclidean")
    labels = db.fit_predict(v_inact.reshape(-1, 1))
    
    inact_mags = np.max(np.abs(acts_matrix[:, inactive_pool]), axis=0)
    inact_vars = var_all[inactive_pool]
    super_mask = (labels == -1) | (inact_mags >= np.quantile(inact_mags, 0.99)) | (inact_vars >= np.quantile(inact_vars, 0.99))
    
    super_coords = inactive_pool[super_mask]
    clustered_coords = inactive_pool[~super_mask]
    
    chunk_list = []
    unique_labs = [l for l in np.unique(labels) if l != -1]
    for lab in unique_labs:
        c_sub_idx = np.where((labels == lab) & (~super_mask))[0]
        if len(c_sub_idx) == 0:
            continue
        c_coords = inactive_pool[c_sub_idx]
        sorted_c = c_coords[np.argsort(v_all[c_coords])]
        num_full = len(sorted_c) // INACT_CHUNK_SIZE
        for ci in range(num_full):
            chunk_list.append(sorted_c[ci * INACT_CHUNK_SIZE : (ci + 1) * INACT_CHUNK_SIZE])
            if len(chunk_list) >= INACT_NUM_CHUNKS:
                break
        if len(chunk_list) >= INACT_NUM_CHUNKS:
            break
            
    if len(chunk_list) < INACT_NUM_CHUNKS:
        assigned = set(np.concatenate(chunk_list) if chunk_list else [])
        avail = [c for c in clustered_coords if c not in assigned]
        needed = INACT_NUM_CHUNKS - len(chunk_list)
        for ci in range(needed):
            if len(avail) >= INACT_CHUNK_SIZE:
                chunk_list.append(np.array(avail[:INACT_CHUNK_SIZE]))
                avail = avail[INACT_CHUNK_SIZE:]
                
    if is_col:
        T_inact = torch.stack([W_clean[:, c].T.float().cpu() for c in chunk_list], dim=0)
    else:
        T_inact = torch.stack([W_clean[c, :].float().cpu() for c in chunk_list], dim=0)
        
    return {
        "tensor": T_inact,
        "chunk_list": chunk_list,
        "super_coords": super_coords,
        "active_pool": active_pool,
        "is_col": is_col,
    }

print("Running inactive DBSCAN pre-clustering across all 26 layers...")
for l in tqdm(range(NUM_LAYERS), desc="Pre-Clustering Inactive"):
    layer_inactive_data[l] = {}
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        is_col = (sub_name == "down_proj")
        layer_inactive_data[l][sub_name] = cluster_inactive_submodule(
            acts_matrix_all[l][sub_name],
            W_orig_all[l][sub_name],
            is_col=is_col,
        )

print("Completed inactive clustering for all 78 submodules on CPU.")


In [ ]:
# =====================================================================
# STEP 5: Define Tucker GD Optimizer & Inactive Evaluation Tiers
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device="cpu"):
    safe_ranks = [
        min(ranks[0], T.shape[0] - 1),
        min(ranks[1], T.shape[1] - 1),
        min(ranks[2], T.shape[2] - 1),
    ]
    core_init, factors_init = tucker(T, rank=safe_ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

inactive_tiers = [
    {
        "name": "Inactive Moderate Tier ([5, 120, 350])",
        "ranks": [5, 120, 350],
    },
    {
        "name": "Inactive Aggressive Tier ([4, 80, 200])",
        "ranks": [4, 80, 200],
    },
    {
        "name": "Inactive Ultra-Aggressive Tier ([3, 40, 100])",
        "ranks": [3, 40, 100],
    },
]

print("Defined Tucker GD optimizer and 3 model-wide inactive tiers.")


In [ ]:
# =====================================================================
# STEP 6: Execute Model-Wide Inactive Sweeps & GLUE MNLI Evaluation
# =====================================================================
tier_benchmarks = []

for tier in inactive_tiers:
    tier_name = tier["name"]
    ranks = tier["ranks"]
    
    print(f"\n{'='*95}")
    print(f"Running Full-Model Inactive Evaluation: {tier_name}")
    print(f"{'='*95}")
    
    tier_gate_errs, tier_up_errs, tier_down_errs = [], [], []
    total_params_saved = 0
    
    # 1. Factorize inactive tensors and inject across all 26 layers
    for l in range(NUM_LAYERS):
        lmod = model.model.layers[l].mlp
        sdata_layer = layer_inactive_data[l]
        
        for sub_name in ["gate_proj", "up_proj", "down_proj"]:
            sdata = sdata_layer[sub_name]
            T_inact = sdata["tensor"]
            
            cg, fg, T_recon, err = optimize_tucker_gd(
                T_inact, ranks=ranks, num_steps=35, lr=1e-3, device="cpu"
            )
            
            if sub_name == "gate_proj": tier_gate_errs.append(err)
            elif sub_name == "up_proj":  tier_up_errs.append(err)
            elif sub_name == "down_proj": tier_down_errs.append(err)
            
            orig_p = T_inact.numel()
            comp_p = cg.numel() + sum(f.numel() for f in fg)
            total_params_saved += (orig_p - comp_p)
            
            # Live injection
            mod_ref = getattr(lmod, sub_name)
            orig_w = W_orig_all[l][sub_name]
            mod_ref.weight.data = orig_w.clone().to(model.device)
            
            if sdata["is_col"]:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[:, c] = T_recon[k].T.to(device=model.device, dtype=mod_ref.weight.dtype)
                if len(sdata["super_coords"]) > 0:
                    mod_ref.weight.data[:, sdata["super_coords"]] = orig_w[:, sdata["super_coords"]].to(model.device)
            else:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[c, :] = T_recon[k].to(device=model.device, dtype=mod_ref.weight.dtype)
                if len(sdata["super_coords"]) > 0:
                    mod_ref.weight.data[sdata["super_coords"], :] = orig_w[sdata["super_coords"], :].to(model.device)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_gate = np.mean(tier_gate_errs) * 100
    mean_up   = np.mean(tier_up_errs) * 100
    mean_down = np.mean(tier_down_errs) * 100
    
    print(f"Layer Factorization Complete across 78 submodules.")
    print(f"  Mean Inactive Recon Errors: gate={mean_gate:.1f}%, up={mean_up:.1f}%, down={mean_down:.1f}%")
    print(f"  Total Inactive Parameters Eliminated: {total_params_saved:,}")

    # 2. Evaluate downstream on GLUE MNLI
    preds, gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {tier_name}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)

            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()

            preds.append(pred_label)
            gts.append(sample["label"])

    acc = accuracy_score(gts, preds)
    delta = acc - baseline_accuracy

    print(f"\nResult for {tier_name}:")
    print(f"  Downstream Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    print(f"  Inactive Params Cut: {total_params_saved:,}")

    tier_benchmarks.append({
        "Variant": tier_name,
        "Ranks": ranks,
        "Mean_Gate_Err": round(mean_gate, 2),
        "Mean_Up_Err": round(mean_up, 2),
        "Mean_Down_Err": round(mean_down, 2),
        "Params_Eliminated": total_params_saved,
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

# Restore pristine model weights across all 26 layers
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        getattr(model.model.layers[l].mlp, sub_name).weight.data = W_orig_all[l][sub_name].clone().to(model.device)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nRestored all 26 layers to pristine weights.")


In [ ]:
# =====================================================================
# STEP 7: Benchmark Summary & JSON Artifact Export
# =====================================================================
print(f"\n{'='*115}")
print(f"{'Variant':<42} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<12} | {'Accuracy':<9} | {'Delta':<8}")
print(f"{'='*115}")
print(f"{'Baseline (Uncompressed)':<42} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<12} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for b in tier_benchmarks:
    print(f"{b['Variant']:<42} | {b['Mean_Gate_Err']:>6.2f}%  | {b['Mean_Up_Err']:>5.2f}%  | {b['Mean_Down_Err']:>6.2f}%   | {b['Params_Eliminated']:<12,d} | {b['Accuracy']:>7.2f}% | {b['Delta']:>+6.2f}%")
print(f"{'='*115}")

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
results_file = artifacts_dir / "14_all_layers_inactive_results.json"

payload = {
    "model_id": MODEL_ID,
    "num_layers": NUM_LAYERS,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "benchmarks": tier_benchmarks,
}

with open(results_file, "w") as f:
    json.dump(payload, f, indent=2)

print(f"\nSaved benchmark results to {results_file}")
